# 03 — Filtros ajustados (sin tocar la API)

Este notebook **reutiliza el CSV `scam_us_raw_dedup.csv`** generado en la tirada diagnóstica y aplica los filtros mejorados.
No hace llamadas a la API → coste 0.

Sirve para validar los filtros nuevos antes de la tirada definitiva. Si los resultados te convencen, los mismos filtros se incorporarán al ETL v3 para la tirada de 20.000 tweets.

## Tres ajustes que aplicamos

1. **Filtro monetario suavizado:** ya no se exige a *todos* los tweets. Tweets que contengan términos muy específicos de fraude (`scam`, `ponzi`, `phishing`, `rug pull`, etc.) pasan aunque no mencionen dinero — eso recupera falsos negativos como "IRS SCAM phonecall".
2. **Filtro de recovery scammers:** descartamos tweets que sigan el patrón típico de estafadores que ofrecen "recuperar fondos" (DM, "we are tracing", "get your money back", listas de marcas con hashtags).
3. **Filtro de noticias/blog promos:** descartamos tweets que parecen titulares de prensa o promos (URL + título sin contenido propio).


In [ ]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

# Cargar el CSV ya descargado en la diagnóstica
df_dedup = pd.read_csv("../data/raw/scam_us_raw_dedup.csv")
print(f"Tweets cargados (deduplicados): {len(df_dedup)}")
print(f"Columnas: {list(df_dedup.columns)}")


## Definición de patrones y filtros

In [ ]:
# -----------------------------------------------------
# AJUSTE 1: Términos de fraude SUFICIENTES (no exigen contexto monetario adicional)
# -----------------------------------------------------
STRONG_FRAUD_TERMS = re.compile(
    r"\b(scam|scammed|scammer|scammers|"
    r"fraud|fraudster|defrauded|"
    r"phishing|smishing|vishing|"
    r"ponzi|rug\s*pull|pig\s*butchering|"
    r"identity\s*theft|wire\s*fraud|"
    r"impersonat|spoof|"
    r"fake\s*(?:text|email|delivery|invoice|website|caller))\b",
    re.IGNORECASE,
)

# Términos monetarios (los originales)
MONEY_TERMS = re.compile(
    r"(\b(money|cash|dollar|dollars|usd|paid|payment|sent|wired|wire|"
    r"transfer|refund|bank|account|credit\s*card|debit|wallet|invoice|"
    r"charged|stolen|lost|victim|defrauded|deposit|withdraw|"
    r"check|venmo|zelle|paypal|crypto|bitcoin|coinbase|gift\s*card)\b|"
    r"\$\s*\d+|\d+\s*(k|dollars|usd))",
    re.IGNORECASE,
)

# -----------------------------------------------------
# AJUSTE 2: Recovery scammers (estafadores que ofrecen recuperar fondos)
# -----------------------------------------------------
RECOVERY_SCAM_PATTERNS = re.compile(
    r"\b("
    r"DM\s+(now|me|us|asap)|"
    r"we\s+are\s+tracing|"
    r"get\s+your\s+money\s+back|"
    r"recover(y|\s+expert|\s+specialist)|"
    r"funds\s+recovery|"
    r"crypto\s+recovery|"
    r"contact\s+us\s+(?:now|asap|today)|"
    r"reach\s+out\s+to\s+(?:us|me)|"
    r"100%\s+(?:guaranteed|recovery|success)|"
    r"lost\s+(?:money|funds|crypto|bitcoin)\s+(?:on|to)\s+\w+\s+(?:scam|scammer)"
    r")\b",
    re.IGNORECASE,
)

# Patrón secundario: lista de marcas como anuncio
BRAND_LIST_PATTERN = re.compile(
    r"(nft\s+scam|bitcoin\s+scam|forex|cfd|binary\s+option|"
    r"wallet\s+drained|bored\s*apes?|ethereum\s+scam)",
    re.IGNORECASE,
)

def is_recovery_scammer(text: str) -> bool:
    """Detecta recovery scammers: patrón principal O lista de marcas + DM/contact."""
    if not isinstance(text, str):
        return False
    if RECOVERY_SCAM_PATTERNS.search(text):
        return True
    # Si menciona DM/contact + 2+ tipos de scam en lista → casi seguro recovery scammer
    has_contact = bool(re.search(r"\b(DM|dm)\b|contact|reach\s+out", text, re.IGNORECASE))
    brand_hits = len(BRAND_LIST_PATTERN.findall(text))
    return has_contact and brand_hits >= 2

# -----------------------------------------------------
# AJUSTE 3: Noticias / blog promos
# -----------------------------------------------------
# Patrón típico de titular: pocas palabras únicas + URL + opcional hashtags
NEWS_HEADLINE_PATTERN = re.compile(
    r"^\W*(leaders?|chief|ceo|president|founder|judge|jury|court|"
    r"police|sentence[d]?|charged|arrested|indicted|"
    r"\d+\s+(?:years?|months?)\s+in\s+prison)\b",
    re.IGNORECASE,
)

PROMO_BLOG_PATTERN = re.compile(
    r"\b("
    r"our\s+(?:recent\s+)?(?:blog|article|post)|"
    r"breaks?\s+down|"
    r"read\s+(?:more|the\s+full)|"
    r"link\s+in\s+bio|"
    r"check\s+(?:out|it)\s+(?:our|this)|"
    r"new\s+(?:blog|article|episode|video)"
    r")\b",
    re.IGNORECASE,
)

def looks_like_news_or_promo(text: str, n_urls: int = 0) -> bool:
    if not isinstance(text, str):
        return False
    if PROMO_BLOG_PATTERN.search(text):
        return True
    # Si el tweet empieza con patrón de titular y tiene URL → es prensa
    if n_urls > 0 and NEWS_HEADLINE_PATTERN.search(text):
        return True
    return False


## Filtros que mantenemos del notebook anterior

In [ ]:
# Patrones del v2 que mantenemos sin cambios
METAPHOR_PATTERNS = re.compile(
    r"\b(life is a scam|love is a scam|system is a scam|"
    r"capitalism is a scam|college is a scam|dating is a scam|"
    r"my life is a scam|everything is a scam|"
    r"life'?s a scam|love'?s a scam)\b",
    re.IGNORECASE,
)

SOFT_POLITICS_PATTERNS = re.compile(
    r"\b(harris|desantis|newsom|kamala|obama|congress|senate|senator|"
    r"republican|democrat|gop|maga|lawmaker|politician|"
    r"voter fraud|election fraud|stolen election|rigged election|"
    r"deep state)\b",
    re.IGNORECASE,
)

BRAND_USERNAMES = {
    "lifelock", "aura", "norton", "nortononline", "mcafee",
    "cnn", "foxnews", "msnbc", "nytimes", "wsj", "reuters",
    "ap", "bbcworld", "abc", "abcnews", "nbcnews", "cbsnews",
}

EMOTIONAL_ONLY = re.compile(
    r"\b(catfish|catfished|catfishing|emotional scam|"
    r"scammed my heart|broke my heart)\b",
    re.IGNORECASE,
)

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_length(text):
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

us_keywords = [
    "united states", "usa", "u.s.", " us ", " us,", "america",
    "new york", "california", "texas", "florida", "chicago",
    "los angeles", "boston", "seattle", "miami", "atlanta",
    "dallas", "houston", "philadelphia", "phoenix", "denver",
    "washington", "san francisco", "san diego",
]

def looks_us(row):
    if row.get("place_country_code") == "US":
        return True
    loc = " " + str(row.get("user_location") or "").lower() + " "
    return any(k in loc for k in us_keywords)


## Aplicación de todos los filtros y resumen

In [ ]:
df = df_dedup.copy()

# Columnas auxiliares (las del v2)
df["clean_text"]       = df["text"].apply(clean_for_length)
df["clean_len"]        = df["clean_text"].str.len()
df["n_words"]          = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["hashtag_ratio"]    = (df["n_hashtags"] / df["n_words"].replace(0, 1)).fillna(0)
df["mention_ratio"]    = (df["n_mentions"] / df["n_words"].replace(0, 1)).fillna(0)
df["is_metaphor"]      = df["text"].fillna("").apply(lambda t: bool(METAPHOR_PATTERNS.search(t)))
df["is_soft_politics"] = df["text"].fillna("").apply(lambda t: bool(SOFT_POLITICS_PATTERNS.search(t)))
df["is_emotional"]     = df["text"].fillna("").apply(lambda t: bool(EMOTIONAL_ONLY.search(t)))
df["is_brand_account"] = df["username"].fillna("").str.lower().isin(BRAND_USERNAMES)
df["likely_us"]        = df.apply(looks_us, axis=1)

# Columnas nuevas v3
df["has_strong_fraud"] = df["text"].fillna("").apply(lambda t: bool(STRONG_FRAUD_TERMS.search(t)))
df["has_money"]        = df["text"].fillna("").apply(lambda t: bool(MONEY_TERMS.search(t)))
df["is_recovery_scammer"] = df["text"].fillna("").apply(is_recovery_scammer)
df["is_news_or_promo"]    = df.apply(lambda r: looks_like_news_or_promo(r["text"], r.get("n_urls", 0)), axis=1)

# CRITERIO RELEVANCIA SEMÁNTICA: tiene términos fuertes de fraude O términos monetarios
df["semantically_relevant"] = df["has_strong_fraud"] | df["has_money"]

print("=== DISTRIBUCIÓN DE FLAGS ===")
print(f"Total tweets:                  {len(df)}")
print(f"Con términos fuertes fraude:   {df['has_strong_fraud'].sum()} ({df['has_strong_fraud'].mean()*100:.1f}%)")
print(f"Con contexto monetario:        {df['has_money'].sum()} ({df['has_money'].mean()*100:.1f}%)")
print(f"Semánticamente relevantes:     {df['semantically_relevant'].sum()} ({df['semantically_relevant'].mean()*100:.1f}%)")
print(f"Recovery scammers detectados:  {df['is_recovery_scammer'].sum()} ({df['is_recovery_scammer'].mean()*100:.1f}%)")
print(f"Noticias / blog promos:        {df['is_news_or_promo'].sum()} ({df['is_news_or_promo'].mean()*100:.1f}%)")
print(f"Metáfora:                      {df['is_metaphor'].sum()}")
print(f"Política suave:                {df['is_soft_politics'].sum()}")
print(f"Emocional puro:                {df['is_emotional'].sum()}")
print(f"Cuentas-marca:                 {df['is_brand_account'].sum()}")
print(f"Probablemente US:              {df['likely_us'].sum()} ({df['likely_us'].mean()*100:.1f}%)")


In [ ]:
# Aplicar todos los filtros
mask = (
    (df["clean_len"] >= 40) &
    (df["semantically_relevant"]) &              # AJUSTE 1: monetario O fraude fuerte
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"]) &
    (~df["is_recovery_scammer"]) &               # AJUSTE 2: nuevos
    (~df["is_news_or_promo"])                    # AJUSTE 3: nuevos
)

df_clean_v3 = df[mask].reset_index(drop=True)

print("=== RESUMEN DE LIMPIEZA v3 ===")
print(f"Brutos deduplicados:      {len(df)}")
print(f"Limpios finales (v3):     {len(df_clean_v3)}")
print(f"Tasa de retención:        {len(df_clean_v3) / max(len(df), 1) * 100:.1f}%")
print()
print("Por categoría temática:")
for label in ["phishing", "payment_apps", "crypto", "romance_monetary", "impersonation"]:
    n = df_clean_v3["query_labels"].fillna("").str.contains(label).sum()
    print(f"  {label:<20} {n:>6}")


## Inspección de resultados nuevos

Vamos a comparar con el v2 para ver qué hemos ganado y qué hemos perdido.


In [ ]:
# Ver qué tweets pasan AHORA que antes no (gracias al ajuste 1: relajar el monetario)
mask_v2 = (
    (df["clean_len"] >= 40) &
    (df["has_money"]) &
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"])
)
df_v2 = df[mask_v2]

newly_included = df_clean_v3[~df_clean_v3["tweet_id"].isin(df_v2["tweet_id"])]
print(f"=== TWEETS RECUPERADOS por el ajuste monetario suavizado: {len(newly_included)} ===")
print("(Tweets que antes se descartaban por no tener palabra monetaria, pero tienen término fuerte de fraude)\n")
for _, row in newly_included.sample(min(10, len(newly_included)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']}")
    print(f"  {row['text'][:280]}")
    print()


In [ ]:
# Ver qué descartamos AHORA (recovery scammers + noticias)
recovery_examples = df[df["is_recovery_scammer"]]
print(f"=== RECOVERY SCAMMERS DETECTADOS: {len(recovery_examples)} ===\n")
for _, row in recovery_examples.sample(min(8, len(recovery_examples)), random_state=42).iterrows():
    print(f"@{row['username']}")
    print(f"  {row['text'][:300]}")
    print()


In [ ]:
news_examples = df[df["is_news_or_promo"]]
print(f"=== NOTICIAS / BLOG PROMOS DETECTADOS: {len(news_examples)} ===\n")
for _, row in news_examples.sample(min(8, len(news_examples)), random_state=42).iterrows():
    print(f"@{row['username']}")
    print(f"  {row['text'][:300]}")
    print()


In [ ]:
# Muestra final de los tweets limpios v3
print("=== MUESTRA ALEATORIA DE 20 TWEETS LIMPIOS v3 ===\n")
for _, row in df_clean_v3.sample(min(20, len(df_clean_v3)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {row['text'][:280]}")
    print()


## Guardado del corpus limpio v3

In [ ]:
import os
os.makedirs("../data/raw", exist_ok=True)
df_clean_v3.to_csv("../data/raw/scam_us_clean_v3.csv", index=False, encoding="utf-8")
print(f"Guardado: ../data/raw/scam_us_clean_v3.csv ({len(df_clean_v3)} filas)")
